In [ ]:
!pip install flask==2.3.3 opencv-python==4.9.0.80 mediapipe==0.10.14 numpy==1.26.4 python-dotenv==1.0.1 requests==2.31.0 gtts==2.5.3 pyngrok==7.1.6 protobuf>=3.20.3 sounddevice==0.5.1
!apt-get update && apt-get install -y ffmpeg libsm6 libxext6

In [ ]:
# Restart runtime to apply new packages
import os
os.kill(os.getpid(), 9)

In [ ]:
import sys
assert sys.version_info[:2] == (3, 10), "Use Python 3.10"
import os
from pyngrok import ngrok
from flask import Flask, request, Response, jsonify
from process_frame import process_frame, add_face
from dotenv import load_dotenv

app = Flask(__name__)
load_dotenv()

@app.route("/video_feed")
def video_feed():
    def generate():
        for frame_bytes in process_frame():
            yield (b"--frame\r\nContent-Type: image/jpeg\r\n\r\n" + frame_bytes + b"\r\n")
    return Response(generate(), mimetype="multipart/x-mixed-replace; boundary=frame")

@app.route("/add_face", methods=["POST"])
def add_face_route():
    try:
        name = request.form["name"]
        image = request.files["image"]
        success, error = add_face(name, image.read())
        return jsonify({"success": success, "error": error}), 200 if success else 400
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

if __name__ == "__main__":
    ngrok.set_auth_token("your_ngrok_token")
    public_url = ngrok.connect(5000).public_url
    print(f"API at: {public_url}")
    app.run(host="0.0.0.0", port=5000)